# Matrix-valued transport coupling

This notebook generates `fig:matrix-valued-transport-coupling`.  A matrix-valued measure attaches a positive semidefinite matrix $A_i\succeq0$ to each spatial location $x_i$.  The trace gives scalar mass, while the ellipse of $A_i/\operatorname{tr}(A_i)$ displays an internal covariance-like state transported together with the mass.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.patches import Ellipse

NAME = "matrix-valued-transport-coupling"
out = figure_dir(NAME)


We draw three source fibers and three target fibers.  The coupling is sparse and purely schematic: it emphasizes that a transported object now has both a spatial position and an internal matrix state.


In [ ]:
src = np.array([[-1.10, 0.55], [-1.18, -0.05], [-0.90, -0.58]])
tgt = np.array([[0.92, 0.62], [1.18, 0.02], [0.98, -0.65]])
As = [
    np.array([[0.16, 0.07], [0.07, 0.06]]),
    np.array([[0.08, -0.035], [-0.035, 0.14]]),
    np.array([[0.13, 0.00], [0.00, 0.05]]),
]
Bs = [
    np.array([[0.10, -0.045], [-0.045, 0.12]]),
    np.array([[0.17, 0.055], [0.055, 0.07]]),
    np.array([[0.06, 0.018], [0.018, 0.15]]),
]
source_weights = np.array([np.trace(A) for A in As])
target_weights = np.array([np.trace(B) for B in Bs])
P = np.array([
    [0.15, 0.05, 0.00],
    [0.00, 0.10, 0.11],
    [0.02, 0.06, 0.14],
])
pairs = [(i, j, P[i, j]) for i in range(3) for j in range(3) if P[i, j] > 0.025]
all_points = np.vstack([src, tgt])
xlim, ylim = padded_limits(all_points, pad=0.23)


In [ ]:
def draw_matrix_atom(ax, center, Sigma, color, mass, zorder=3):
    vals, vecs = np.linalg.eigh(Sigma / np.trace(Sigma))
    order = np.argsort(vals)[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    scale = 0.78 * np.sqrt(mass / max(source_weights.max(), target_weights.max()))
    ell = Ellipse(
        center,
        width=scale * np.sqrt(vals[0]),
        height=scale * np.sqrt(vals[1]),
        angle=angle,
        facecolor=color,
        edgecolor=color,
        lw=0.85,
        alpha=0.18,
        zorder=zorder,
    )
    ax.add_patch(ell)
    ax.add_patch(Ellipse(center, ell.width, ell.height, angle=angle, facecolor="none", edgecolor=color, lw=0.85, zorder=zorder + 1))
    ax.scatter([center[0]], [center[1]], s=DIRAC_MARKER_SIZE * (0.8 + mass), marker="o", color=color, edgecolor="none", linewidth=0, zorder=zorder + 2)

def decorate(ax):
    ax.set_aspect("equal")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    remove_axes(ax)

fig, ax = plt.subplots(figsize=(2.50, 2.05))
for z, A, w in zip(src, As, source_weights):
    draw_matrix_atom(ax, z, A, RED, w)
for z, B, w in zip(tgt, Bs, target_weights):
    draw_matrix_atom(ax, z, B, BLUE, w)
decorate(ax)
save_pdf(fig, out / "matrix-masses.pdf", pad_inches=0.06)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.50, 2.05))
draw_transport_segments(ax, src, tgt, pairs, color=VIOLET, max_width=1.55, alpha_scale=0.55, zorder=1)
for z, A, w in zip(src, As, source_weights):
    draw_matrix_atom(ax, z, A, RED, w)
for z, B, w in zip(tgt, Bs, target_weights):
    draw_matrix_atom(ax, z, B, BLUE, w)
decorate(ax)
save_pdf(fig, out / "matrix-coupling.pdf", pad_inches=0.06)
plt.close(fig)
